In [1]:
import os
import sys
import json
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from src.inference import CLIPInference
from src.aggregation import aggregate_video_scores
from src.metrics import evaluate

In [2]:
with open("../artifacts/threshold.json", "r") as f:
    best_threshold = json.load(f)["threshold"]

print("Loaded threshold:", best_threshold)

Loaded threshold: 0.01629638671875


In [3]:
manifest = pd.read_csv("../data/segments_manifest_16.csv")

test_df = manifest[manifest["split"] == "test"].copy()

print("Test segments:", len(test_df))
test_df.head()

Test segments: 307


,segment_uid,split,class,video_id,segment_id,num_frames,segment_path
519,test_Abuse_Abuse028_x264_seg_0000,test,Abuse,Abuse028_x264,seg_0000,16,test/Abuse/Abuse028_x264/seg_0000
520,test_Abuse_Abuse028_x264_seg_0001,test,Abuse,Abuse028_x264,seg_0001,16,test/Abuse/Abuse028_x264/seg_0001
521,test_Abuse_Abuse028_x264_seg_0002,test,Abuse,Abuse028_x264,seg_0002,16,test/Abuse/Abuse028_x264/seg_0002
522,test_Abuse_Abuse028_x264_seg_0003,test,Abuse,Abuse028_x264,seg_0003,16,test/Abuse/Abuse028_x264/seg_0003
523,test_Abuse_Abuse028_x264_seg_0004,test,Abuse,Abuse028_x264,seg_0004,16,test/Abuse/Abuse028_x264/seg_0004


In [4]:
from src.config import ANOMALY_CLASSES, NORMAL_CLASSES

model = CLIPInference(scoring_mode="max_mean")

TEXTS = ANOMALY_CLASSES + NORMAL_CLASSES
model.set_text_prompts(TEXTS)

print("Model ready with config prompts.")

Model ready with config prompts.


In [5]:
from tqdm import tqdm
import os

results_test = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    segment_path = row["segment_path"]

    full_path = os.path.join("..", "data", "segments_16", segment_path)

    prediction = model.predict_segment(full_path)

    results_test.append({
        "video_id": row["video_id"],
        "segment_uid": row["segment_uid"],
        "true_class": row["class"],
        "score": prediction["score"]
    })

results_test_df = pd.DataFrame(results_test)

print("Processed segments:", len(results_test_df))
results_test_df.head()

100%|███████████████████████████████████████████████████████████████| 307/307 [00:36<00:00,  8.46it/s]

Processed segments: 307


,video_id,segment_uid,true_class,score
0,Abuse028_x264,test_Abuse_Abuse028_x264_seg_0000,Abuse,0.031006
1,Abuse028_x264,test_Abuse_Abuse028_x264_seg_0001,Abuse,0.028687
2,Abuse028_x264,test_Abuse_Abuse028_x264_seg_0002,Abuse,0.032837
3,Abuse028_x264,test_Abuse_Abuse028_x264_seg_0003,Abuse,0.033203
4,Abuse028_x264,test_Abuse_Abuse028_x264_seg_0004,Abuse,0.037354


In [6]:
from src.config import AGGREGATION_METHOD

video_test_df = aggregate_video_scores(
    results_test_df,
    method=AGGREGATION_METHOD
)

print("Total test videos:", len(video_test_df))
video_test_df.head()

Total test videos: 14


,video_id,score,true_class,is_anomaly
0,Abuse028_x264,0.034448,Abuse,True
1,Arrest001_x264,0.044604,Arrest,True
2,Arson007_x264,0.085669,Arson,True
3,Assault006_x264,0.052832,Assault,True
4,Burglary005_x264,0.088538,Burglary,True


In [7]:
metrics_test = evaluate(video_test_df, threshold=best_threshold)

print("---- TEST METRICS ----")
for k, v in metrics_test.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")

---- TEST METRICS ----
accuracy: 0.9286
precision: 0.9286
recall: 1.0000
f1: 0.9630
auc: 1.0000
used_threshold: 0.0163


In [8]:
cm = metrics_test["confusion_matrix"]

tn = cm["tn"]
fp = cm["fp"]
fn = cm["fn"]
tp = cm["tp"]

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

TN: 0
FP: 1
FN: 0
TP: 13


In [9]:
# Prediction üret
video_test_df["predicted"] = (video_test_df["score"] > best_threshold)

# False positives
false_positives = video_test_df[
    (video_test_df["is_anomaly"] == False) &
    (video_test_df["predicted"] == True)
]

false_positives

,video_id,score,true_class,is_anomaly,predicted
7,Normal_Videos_003_x264,0.030811,NormalVideos,False,True


In [10]:
# O videonun segment skorlarını görelim
fp_video_id = "Normal_Videos_003_x264"

fp_segments = results_test_df[
    results_test_df["video_id"] == fp_video_id
].sort_values("score", ascending=False)

fp_segments

,video_id,segment_uid,true_class,score
212,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0009,NormalVideos,0.031372
217,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0014,NormalVideos,0.031250
216,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0013,NormalVideos,0.030518
218,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0015,NormalVideos,0.030273
211,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0008,NormalVideos,0.030273
210,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0007,NormalVideos,0.030151
219,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0016,NormalVideos,0.030029
215,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0012,NormalVideos,0.030029
213,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0010,NormalVideos,0.030029
207,Normal_Videos_003_x264,test_NormalVideos_Normal_Videos_003_x264_seg_0004,NormalVideos,0.029907
